# Sketch2Shape AI — Integrated Inference

End-to-end Colab notebook: **sketch → preprocessing → ControlNet → TripoSR → GLB**. Tested component settings from the hackathon experiments: SD1.5 + Scribble ControlNet, TripoSR, CPU marching-cubes fallback, and `--foreground-ratio 0.95`.

**Run from top to bottom on a Colab T4 GPU.** Each runtime starts with temporary `/content/` storage; model weights may need to download again. The two models use separate Python 3.10 environments and run sequentially to reduce GPU memory overlap. The final entry point is `generate_3d(sketch_path, prompt) -> str`.

External pretrained components: `lllyasviel/control_v11p_sd15_scribble`, `stable-diffusion-v1-5/stable-diffusion-v1-5`, `VAST-AI-Research/TripoSR`, and `stabilityai/TripoSR`. See each source's licensing and attribution before submission. GPU inference is not executed during offline cleanup; do a fresh **Runtime → Run all** test before submission.

In [ ]:
import subprocess
import torch

assert torch.cuda.is_available(), "Select a GPU runtime in Colab"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.10"], check=True)

In [ ]:
import subprocess
from pathlib import Path
for name in ("controlnet-env", "triposr-env"):
    python = Path(f"/content/{name}/bin/python")
    if not python.exists():
        subprocess.run(["uv", "venv", f"/content/{name}", "--python", "3.10"], check=True)
    assert python.exists()
    print(name, subprocess.run([str(python), "--version"], capture_output=True, text=True, check=True).stdout.strip())

In [ ]:
import subprocess
for env in ["controlnet-env", "triposr-env"]:
    subprocess.run([
        "uv", "pip", "install", "--python", f"/content/{env}/bin/python",
        "torch==2.1.2", "torchvision==0.16.2",
        "--index-url", "https://download.pytorch.org/whl/cu121"
    ], check=True)

In [ ]:
import subprocess
for name in ("controlnet-env", "triposr-env"):
    subprocess.run([
        f"/content/{name}/bin/python", "-c",
        "import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0))"
    ], check=True)

In [ ]:
import subprocess
subprocess.run([
    "uv", "pip", "install", "--python", "/content/controlnet-env/bin/python",
    "numpy==1.26.4", "diffusers==0.27.2", "transformers==4.38.2",
    "accelerate==0.29.3", "huggingface-hub==0.25.2", "safetensors", "Pillow==10.2.0"
], check=True)

In [ ]:
import subprocess
subprocess.run([
    "/content/controlnet-env/bin/python", "-c",
    "import diffusers,transformers,accelerate; print(diffusers.__version__,transformers.__version__,accelerate.__version__)"
], check=True)

In [ ]:
import subprocess
from pathlib import Path
repo = Path("/content/TripoSR")
if not (repo / "run.py").is_file():
    if repo.exists():
        raise RuntimeError("TripoSR directory exists without run.py; inspect it before cloning")
    subprocess.run(["git", "clone", "https://github.com/VAST-AI-Research/TripoSR.git", str(repo)], check=True)
print("TripoSR:", (repo / "run.py").exists())

In [ ]:
import subprocess
from pathlib import Path

lines = Path("/content/TripoSR/requirements.txt").read_text().splitlines()
lines = [line for line in lines if "torchmcubes" not in line and line.strip() != "gradio" and not line.startswith("huggingface-hub")]
lines += [
    "huggingface-hub==0.17.3",
    "numpy==1.26.4",
    "scikit-image==0.22.0",
    "onnxruntime-gpu"
]
Path("/content/triposr-req.txt").write_text("\n".join(lines) + "\n")
subprocess.run([
    "uv", "pip", "install", "--python", "/content/triposr-env/bin/python", "-r", "/content/triposr-req.txt"
], check=True)

In [ ]:
from pathlib import Path

path = Path("/content/TripoSR/tsr/models/isosurface.py")
source = path.read_text()

old = "from torchmcubes import marching_cubes"

replacement = """
from skimage.measure import marching_cubes as _skimage_mc

def marching_cubes(volume, level):
    verts, faces, _, _ = _skimage_mc(
        volume.detach().float().cpu().numpy(),
        level=level
    )

    verts = np.ascontiguousarray(verts[:, ::-1])
    faces = np.ascontiguousarray(faces.astype(np.int64))

    return torch.from_numpy(verts), torch.from_numpy(faces)
"""

if old in source:
    path.write_text(source.replace(old, replacement))
elif "_skimage_mc" not in source:
    raise RuntimeError("Unexpected isosurface.py content")

print("Marching cubes patched")

In [ ]:
import subprocess

subprocess.run([
    "/content/triposr-env/bin/python",
    "-c",
    """
import numpy as np
import torch
from tsr.models.isosurface import marching_cubes

x, y, z = np.indices((16, 16, 16))
volume = ((x-8)**2 + (y-8)**2 + (z-8)**2 - 25).astype(np.float32)

vertices, faces = marching_cubes(torch.from_numpy(volume), 0.0)

assert len(vertices) > 0
assert len(faces) > 0

print("Vertices:", len(vertices))
print("Faces:", len(faces))
print("Mesh extraction passed")
"""
], cwd="/content/TripoSR", check=True)

In [ ]:
%pip -q install trimesh Pillow numpy

## Upload a sketch

Use a simple dark-on-light drawing. The function below crops and converts it into the ControlNet input automatically. An optional text prompt describes its materials and appearance.

In [ ]:
from google.colab import files
from pathlib import Path
from PIL import Image

uploaded = files.upload()
assert uploaded, "Upload a sketch"
sketch_path = Path("/content") / next(iter(uploaded))
display(Image.open(sketch_path))
print("Sketch:", sketch_path)

## Create the generation script

This script runs only in `controlnet-env`; the notebook process handles sketch preprocessing and GLB validation.

In [ ]:
%%writefile /content/generate_image.py

import argparse
import torch
from PIL import Image
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    UniPCMultistepScheduler
)

parser = argparse.ArgumentParser()
parser.add_argument("--control", required=True)
parser.add_argument("--output", required=True)
parser.add_argument("--prompt", required=True)
args = parser.parse_args()

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_scribble",
    torch_dtype=torch.float16
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
)

pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config
)

pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()

image = pipe(
    prompt=args.prompt,
    negative_prompt="multiple objects, people, text, watermark, cropped, distorted",
    image=Image.open(args.control).convert("RGB"),
    num_inference_steps=20,
    guidance_scale=7.0,
    controlnet_conditioning_scale=1.1,
    generator=torch.Generator(device="cpu").manual_seed(42)
).images[0]

image.save(args.output)

## Reusable end-to-end entry point

Returns the path of an exported GLB. Generates job-specific files under `/content/outputs/<job-id>/`. Inference on the T4 is sequential, so don't run concurrent requests without a queue.

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps
import numpy as np
import subprocess
import uuid

def generate_3d(sketch_path, prompt="a single realistic object"):
    job = uuid.uuid4().hex[:10]
    out = Path("/content/outputs") / job
    out.mkdir(parents=True, exist_ok=True)

    sketch = ImageOps.autocontrast(
        Image.open(sketch_path).convert("L")
    )

    mask = np.asarray(sketch) < 200
    if not mask.any():
        raise ValueError("No sketch lines detected")

    ys, xs = np.where(mask)
    sketch = sketch.crop((
        xs.min(), ys.min(),
        xs.max() + 1, ys.max() + 1
    ))

    sketch.thumbnail((440, 440))

    canvas = Image.new("L", (512, 512), 255)
    canvas.paste(sketch, (
        (512 - sketch.width) // 2,
        (512 - sketch.height) // 2
    ))

    control = Image.fromarray(
        np.where(np.asarray(canvas) < 200, 255, 0).astype("uint8")
    ).convert("RGB")

    control_path = out / "control.png"
    image_path = out / "generated.png"

    control.save(control_path)

    subprocess.run([
        "/content/controlnet-env/bin/python",
        "/content/generate_image.py",
        "--control", str(control_path),
        "--output", str(image_path),
        "--prompt", prompt
    ], check=True)

    subprocess.run([
        "/content/triposr-env/bin/python",
        "run.py",
        str(image_path),
        "--output-dir", str(out / "reconstruction"),
        "--model-save-format", "glb",
        "--foreground-ratio", "0.95",
        "--mc-resolution", "192",
        "--chunk-size", "4096"
    ], cwd="/content/TripoSR", check=True)

    raw = out / "reconstruction/0/mesh.glb"
    final = out / "model.glb"

    import trimesh

    scene = trimesh.load(raw, force="scene")

    if not scene.geometry:
        raise ValueError("Empty mesh")

    for mesh in scene.geometry.values():
        if mesh.is_watertight and mesh.volume < 0:
            mesh.invert()

    scene.export(final)

    verified = trimesh.load(final, force="scene")
    meshes = list(verified.geometry.values())

    if not meshes or not all(
        len(m.vertices) > 0 and len(m.faces) > 0
        for m in meshes
    ):
        raise ValueError("Invalid exported mesh")

    if verified.bounds is None or not np.isfinite(verified.bounds).all():
        raise ValueError("Invalid mesh bounds")
    if not np.all(verified.extents > 0):
        raise ValueError("Degenerate 3D mesh")

    return str(final)

## End-to-end test

Start with one sketch and inspect both the generated image and the mesh. Change `prompt` for different objects.

In [ ]:
prompt = (
    "single wooden dining chair, isolated studio product photograph, "
    "four distinct legs, tall open backrest with three horizontal slats, "
    "complete chair visible, three-quarter front view, "
    "plain light gray background, no floor, no wall, no shadows, "
    "accurate geometry, realistic wooden material"
)
model_path = generate_3d(str(sketch_path), prompt)
print(model_path)

In [ ]:
from pathlib import Path
from PIL import Image

out = Path(model_path).parent
print("Control image:")
display(Image.open(out / "control.png"))
print("Generated object:")
display(Image.open(out / "generated.png"))
processed = out / "reconstruction" / "0" / "input.png"
if processed.is_file():
    print("TripoSR foreground:")
    display(Image.open(processed))

In [ ]:
from pathlib import Path
import trimesh
import numpy as np

assert Path(model_path).is_file()
scene = trimesh.load(model_path, force="scene")
meshes = list(scene.geometry.values())
assert meshes and all(len(m.vertices) > 0 and len(m.faces) > 0 for m in meshes)
assert np.isfinite(scene.bounds).all() and np.all(scene.extents > 0)
print("Vertices:", sum(len(m.vertices) for m in meshes))
print("Faces:", sum(len(m.faces) for m in meshes))
print("Components:", sum(len(m.split()) for m in meshes))
print("Extents:", scene.extents)
print("Watertight:", all(m.is_watertight for m in meshes))
print("Colors:", [m.visual.kind for m in meshes])

In [ ]:
from google.colab import files
files.download(model_path)

## FastAPI handoff

Backend input: path to an uploaded PNG/JPG plus an optional prompt. Call `generate_3d(str(sketch_path), prompt)` on the GPU host, then serve the GLB returned path. This notebook does **not** configure remote FastAPI access or a job queue; production integration needs a persistent GPU runtime or separate inference service.

For reproducibility, record the actual GPU model, versions, model-weight access requirements, approximate run time, model licenses, dataset sources (if any), and the CPU marching-cubes modification. The GLB is real geometry, but visual fidelity must be inspected rather than inferred from mesh validity alone.